# Tedim Hymn Scraper - Get All Songs and Lyrics
## Source: http://www.tedimhymn.com/

In [ ]:
# Internet check (Kaggle requirement)
import os
import urllib.request

try:
    urllib.request.urlopen('http://www.google.com', timeout=5)
    print('Internet: ON')
except Exception:
    print('Internet: OFF')
    raise SystemExit('No internet connection')

In [ ]:
pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time
import zipfile
import pandas as pd
from pathlib import Path
from IPython.display import FileLink, display

## 1. Fetch the main page

In [ ]:
BASE_URL = 'http://www.tedimhymn.com/'
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_FILE = OUTPUT_DIR / 'tedim_hymns.jsonl'

print('Fetching main page...')
response = requests.get(BASE_URL, timeout=60)
response.encoding = 'utf-8'
print(f'Status: {response.status_code}')
print(f'Size: {len(response.text):,} bytes')

## 2. Parse all hymns from the table

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find('table', id='labutable')
rows = table.find('tbody').find_all('tr')
print(f'Total hymns found: {len(rows)}')

In [ ]:
def clean_lyrics(text):
    if not text:
        return ''
    text = re.sub(r'Key\s*:\s*\w+', '', text).strip()
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

hymns = []

for i, row in enumerate(rows):
    try:
        cells = row.find_all('td')
        if len(cells) < 9:
            continue

        hymn_no = cells[0].get_text(strip=True)
        title = cells[1].get_text(strip=True)

        modal = cells[2].find('div', class_='modal-body')
        lyrics_raw = ''
        if modal:
            p_tag = modal.find('p')
            if p_tag:
                lyrics_raw = p_tag.get_text()
        lyrics = clean_lyrics(lyrics_raw)

        pdf_link = ''
        pdf_a = cells[3].find('a')
        if pdf_a and pdf_a.get('href'):
            pdf_link = BASE_URL.rstrip('/') + '/' + pdf_a['href'].lstrip('/')

        pptx_link = ''
        pptx_a = cells[4].find('a')
        if pptx_a and pptx_a.get('href'):
            pptx_link = BASE_URL.rstrip('/') + '/' + pptx_a['href'].lstrip('/')

        midi_link = ''
        midi_a = cells[5].find('a')
        if midi_a and midi_a.get('href'):
            midi_link = BASE_URL.rstrip('/') + '/' + midi_a['href'].lstrip('/')

        scripture = cells[6].get_text(strip=True)
        composer = cells[7].get_text(strip=True)
        key = cells[8].get_text(strip=True)

        hymns.append({
            'no': hymn_no,
            'title': title,
            'lyrics': lyrics,
            'composer': composer,
            'key': key,
            'scripture': scripture,
            'pdf_url': pdf_link,
            'pptx_url': pptx_link,
            'midi_url': midi_link,
        })

        if (i + 1) % 50 == 0:
            print(f'Parsed {i + 1}/{len(rows)} hymns...')

    except Exception as e:
        print(f'Error parsing row {i + 1}: {e}')
        continue

print(f'\nSuccessfully parsed {len(hymns)} hymns')

## 3. Preview first few hymns

In [ ]:
for hymn in hymns[:3]:
    print(f"No: {hymn['no']}")
    print(f"Title: {hymn['title']}")
    print(f"Composer: {hymn['composer']}")
    print(f"Key: {hymn['key']}")
    print(f"Scripture: {hymn['scripture']}")
    print(f"Lyrics (first 200 chars): {hymn['lyrics'][:200]}...")
    print(f"PDF: {hymn['pdf_url']}")
    print(f"MIDI: {hymn['midi_url']}")
    print('-' * 80)

## 4. Save to JSONL

In [ ]:
print(f'Saving to {OUTPUT_FILE}...')

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for hymn in hymns:
        f.write(json.dumps(hymn, ensure_ascii=False) + '\n')

print(f'Saved {len(hymns)} hymns to {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size:,} bytes')

## 5. Statistics

In [ ]:
df = pd.read_json(OUTPUT_FILE, lines=True)

print('=== Tedim Hymn Dataset Statistics ===')
print(f'Total hymns: {len(df)}')
print(f'\nLyrics stats:')
print(f'  - Hymns with lyrics: {df["lyrics"].str.len().gt(0).sum()}')
print(f'  - Average lyrics length: {df["lyrics"].str.len().mean():.0f} chars')
print(f'  - Min lyrics length: {df["lyrics"].str.len().min()} chars')
print(f'  - Max lyrics length: {df["lyrics"].str.len().max()} chars')
print(f'\nResource links:')
print(f'  - With PDF: {df["pdf_url"].str.len().gt(0).sum()}')
print(f'  - With PPTX: {df["pptx_url"].str.len().gt(0).sum()}')
print(f'  - With MIDI: {df["midi_url"].str.len().gt(0).sum()}')
print(f'\nHymn number range: {df["no"].min()} - {df["no"].max()}')
print(f'\nSample hymn numbers:')
print(df['no'].head(20).tolist())

## 6. Download ALL resources (PDF, PPTX, MIDI)

In [ ]:
PDF_DIR = OUTPUT_DIR / 'tedim_hymns_resources' / 'pdf'
PPTX_DIR = OUTPUT_DIR / 'tedim_hymns_resources' / 'pptx'
MIDI_DIR = OUTPUT_DIR / 'tedim_hymns_resources' / 'midi'

PDF_DIR.mkdir(parents=True, exist_ok=True)
PPTX_DIR.mkdir(parents=True, exist_ok=True)
MIDI_DIR.mkdir(parents=True, exist_ok=True)

stats = {'pdf': {'ok': 0, 'fail': 0, 'skip': 0},
         'pptx': {'ok': 0, 'fail': 0, 'skip': 0},
         'midi': {'ok': 0, 'fail': 0, 'skip': 0}}

def download_file(url, dest_dir):
    if not url:
        return 'skip'
    fname = url.split('/')[-1]
    dest = dest_dir / fname
    if dest.exists():
        return 'skip'
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(dest, 'wb') as f:
                f.write(r.content)
            return 'ok'
        return 'fail'
    except Exception:
        return 'fail'

total = len(hymns)
for i, hymn in enumerate(hymns):
    result = download_file(hymn['pdf_url'], PDF_DIR)
    stats['pdf'][result] += 1
    result = download_file(hymn['pptx_url'], PPTX_DIR)
    stats['pptx'][result] += 1
    result = download_file(hymn['midi_url'], MIDI_DIR)
    stats['midi'][result] += 1
    if (i + 1) % 20 == 0:
        print(f'Downloaded {i + 1}/{total}...')
    time.sleep(0.5)

print(f'\n=== Download Summary ===')
for ftype in ['pdf', 'pptx', 'midi']:
    s = stats[ftype]
    print(f'{ftype.upper()}: {s["ok"]} downloaded, {s["skip"]} skipped, {s["fail"]} failed')

## 7. Create ZIP archive for download

In [ ]:
ZIP_FILE = OUTPUT_DIR / 'tedim_hymns_complete.zip'

print(f'Creating ZIP archive...')

with zipfile.ZipFile(ZIP_FILE, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUTPUT_FILE, 'tedim_hymns.jsonl')
    print(f'  Added: tedim_hymns.jsonl')
    
    pdf_count = 0
    for f in sorted(PDF_DIR.iterdir()):
        zf.write(f, f'pdf/{f.name}')
        pdf_count += 1
    print(f'  Added: {pdf_count} PDFs')
    
    pptx_count = 0
    for f in sorted(PPTX_DIR.iterdir()):
        zf.write(f, f'pptx/{f.name}')
        pptx_count += 1
    print(f'  Added: {pptx_count} PPTXs')
    
    midi_count = 0
    for f in sorted(MIDI_DIR.iterdir()):
        zf.write(f, f'midi/{f.name}')
        midi_count += 1
    print(f'  Added: {midi_count} MIDIs')

zip_size_mb = ZIP_FILE.stat().st_size / (1024 * 1024)
print(f'\nZIP created: {ZIP_FILE.name}')
print(f'Size: {zip_size_mb:.1f} MB')

## 8. Download link

In [ ]:
print('Click the link below to download the complete ZIP file:')
display(FileLink('tedim_hymns_complete.zip'))